In [43]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import TypedDict, Literal

In [31]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [32]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [33]:
class SentimentSchema(BaseModel):
    sentiment : Literal['Positive', 'Negative'] = Field(description="Sentiment of the Review")

In [34]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [39]:
structured_model = llm.with_structured_output(
    SentimentSchema,
    method="json_schema"
)
structured_model2 = llm.with_structured_output(DiagnosisSchema)

In [40]:
prompt = "What  is the sentiment of the review - This Product is Good"
structured_model.invoke(prompt)

SentimentSchema(sentiment='Positive')

In [7]:
class ReviewState(TypedDict):
    review :str
    sentiment : Literal['Positive', 'Negative']
    diagnosis : dict
    response : str

In [48]:
def find_sentiment(state : ReviewState):
    prompt = f"for the following review find the sentiment \n {state['review']}"
    sentiment = structured_model.invoke(prompt)

    return {'sentiment': sentiment}


def check_condition(state : ReviewState) -> Literal['positive_response', 'run_diagnosis']:

    if state['sentiment'] == 'postive' :
        return 'positive_response'
    else :
        return 'run_diagnosis'


def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    
    response = llm.invoke(prompt).content

    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = llm.invoke(prompt).content

    return {'response': response}

In [49]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_condition)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [50]:
initial_state={
    'review': "I've been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(initial_state)

{'review': "I've been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.",
 'sentiment': SentimentSchema(sentiment='Negative'),
 'diagnosis': {'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'},
 'response': 'Hi\u202f[Customer Name],\n\nI’m really sorry you’re experiencing this issue—it’s completely understandable how frustrating that must be, especially when you need things to work right away. Thank you for letting us know, and I’m treating this as a top‑priority case.\n\n**What’s happening:**  \nYou reported a bug with **[brief description of the problem – e.g., “the checkout button not responding on the mobile app”**]. I can see how this would block you from completing your tasks, and I’m committed to getting it fixed as quickly as possible.\n\n**What we’re doing right now:**  \n\n1. **Escalated to our enginee